**Goal of this notebook:**
Eliminate the oracle-mask dependency by strictly aligning DeepLabV3+ with the frozen 5-fold cluster-grouped splits and exporting out-of-fold (OOF) predicted kidney crops for all 1,985 images. For that in this notebook we will:


1 - Verify DeepLabV3+ 5-fold splits match the exact 88-cluster frozen classification folds (zero patient/frame leakage) and save master manifest folds_master.csv.

2 - Run each folds DeepLabV3+ model strictly on its held-out validation fold to generate OOF binary kidney masks for all 1,985 images.

3 - Extract soft-padded kidney bounding crops from the OOF predicted masks and log predicted-vs-ground-truth box IoU statistics.



##**1 - Creating folds_master.csv and conecting scr/kidney**

clona el repo, cruza folds_corrected.csv con nested_cv_splits.csv, verifica que 0 clusters se cruzan entre folds y guarda folds_master.csv

In 1, we clone the repository from git to connect to it and use the notebooks and files inside the 'segmentation' branch, since thats where the segmentation models we tried(and as we saw deeplabv3p had the better results). Here we cross the folds_corrected.csv with nested_cv_splits.csv, verifies that no same clusters in between folds and it saves the folds_master.csv needed.

In [1]:
import os, sys, subprocess
from pathlib import Path
import pandas as pd

# 1. Clonar repo si no existe y situarnos dentro
REPO_URL = "https://github.com/Noadori34/Project-data-science.git"
root = Path("/content/Project-data-science")
if not root.exists():
    subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
os.chdir(root)

# 2. Traer el código de segmentación de la rama 'segmentation'
subprocess.run(["git", "fetch", "origin", "segmentation"], check=True)
subprocess.run(["git", "checkout", "origin/segmentation", "--", "src/kidney", "requirements-seg.txt", "scripts"], check=True)

# 3. Instalar dependencias de segmentación (MONAI, segmentation-models-pytorch, etc.)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements-seg.txt"], check=True)
sys.path.insert(0, str(root / "src"))

# 4. Crear folds_master.csv verificado sin data leakage
folds_cor = pd.read_csv("results/eda/folds_corrected.csv")
nested = pd.read_csv("results/eda/nested_cv_splits.csv")

assert (folds_cor.groupby("group_id")["fold"].nunique() > 1).sum() == 0, "¡Hay fuga de datos entre folds!"

val_map = nested[nested["split"] == "val"].groupby("filename")["outer_fold"].apply(list).to_dict()
folds_master = folds_cor.copy()
folds_master["image_id"] = folds_master["filename"].str.replace(".jpg", "", regex=False)
folds_master["inner_val_in_outer_folds"] = folds_master["filename"].map(lambda f: ",".join(map(str, val_map.get(f, []))))
folds_master.to_csv("results/eda/folds_master.csv", index=False)

# 5. Sobrescribir folds.csv para que el pipeline de src/kidney use automáticamente los folds corregidos
folds_cor.to_csv("results/eda/folds.csv", index=False)
Path("results/splits").mkdir(parents=True, exist_ok=True)
folds_master[["image_id", "filename", "diagnosis", "group_id", "fold"]].to_csv("results/splits/folds.csv", index=False)

print("Entorno listo y folds_master.csv creado con 0% leakage.")
print("\n--- Revisando splits.py y train_seg.py ---")
print(Path("src/kidney/splits.py").read_text())

Entorno listo y folds_master.csv creado con 0% leakage.

--- Revisando splits.py y train_seg.py ---
"""5-fold split, plan.md 1.2.

The plan asks for one `results/splits/folds.csv` (`image_id, fold`) read by every notebook.
The team already committed a leak-free split (`results/eda/folds.csv`) that keeps near-duplicate
groups (references.md, "Our findings") in the same fold. We adopt it as the canonical split, so
copies of one image never sit in train and test at once. If it is missing we fall back to
`StratifiedGroupKFold` / `StratifiedKFold` [S25] with random_state=42, stratified on Global diagnosis.

Per repetition k (as in [24]): test = fold k, validation = fold (k+1) % 5, train = the other three.
"""
from __future__ import annotations

import pandas as pd

from . import config as C


def build_folds(force: bool = False) -> pd.DataFrame:
    C.SPLITS.mkdir(parents=True, exist_ok=True)
    if C.FOLDS_CSV.exists() and not force:
        return read_folds()
    if C.TEAM_FOLDS_CSV.exis

##**2 - Train the chosen model DeepLabV3+ in the 5 fold corrected**

We are training here deeplabv3p_r50, as its the chosen segmentation model, as it gave the best results, doing over the corrected folds, and then it will save the 1.985 masks OOF in results/pred_masks/deeplabv3p_r50/

In [2]:
!PYTHONPATH=src python -m kidney.train_seg --model deeplabv3p_r50

fold roles: {0: {'test': 0, 'val': 1, 'train': [2, 3, 4]}, 1: {'test': 1, 'val': 2, 'train': [0, 3, 4]}, 2: {'test': 2, 'val': 3, 'train': [0, 1, 4]}, 3: {'test': 3, 'val': 4, 'train': [0, 1, 2]}, 4: {'test': 4, 'val': 0, 'train': [1, 2, 3]}}
I0000 00:00:1795369432.178343    1375 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795369435.748664    1375 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataL

##**3 - Generating the 1985 OOF masks,soft padded crops and IoU statistics**

- We use the DeepLabV3+ weights/reutilizing the code in segmentation.ipynb notebook, and we train the folds over the new created folds_master.csv

- As folds end $k \in \{0, 1, 2, 3, 4\}$, the model predicts the masks **only** over its validation fold(zero data leakage), and apply the post-processing. It saves:

results/oof_masks/ (the 1.985 binary masks predicted).

results/oof_crops/ (the 1.985 images cropped with soft-padding over the kidney).

results/eda/oof_segmentation_iou.csv (the table comparing the bounding box and the predicted mask against the real masks .mat, together with the summary of IoU/Dice(Dice counts the area the two share, doubled, and divides it by the total area of both outlines: Dice = 2 × shared area ÷ (our area + doctor's area).
IoU (intersection over union) divides the shared area by everything covered by either outline: IoU = shared area ÷ combined ar Both go from 0 (no overlap at all) to 1 (identical outlines). IoU is the stricter of the two).

In [3]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from kidney import config as C
from kidney.data import list_dataset_a, load_image_a, load_mask_a, mask_to_box

# 1. Crear carpeta para los 1,985 recortes OOF
crops_dir = Path("results/oof_crops")
crops_dir.mkdir(parents=True, exist_ok=True)

PAD_RATIO = 0.15  # Soft-padding del 15% alrededor del riñón predicho (mismo criterio que Phase 2)

def box_iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

folds_master = pd.read_csv("results/eda/folds_master.csv", dtype={"image_id": str})
stats = []

for row in folds_master.itertuples():
    iid = row.image_id
    img = load_image_a(iid)
    gt_mask = load_mask_a(iid)
    pred_mask = (cv2.imread(str(C.PRED_MASKS / "deeplabv3p_r50" / f"{iid}.png"), cv2.IMREAD_GRAYSCALE) > 0).astype(np.uint8)

    h, w = img.shape[:2]
    gt_box = mask_to_box(gt_mask)

    # Si por algún caso extremo la máscara predicha estuviera vacía, fallback a imagen completa
    if pred_mask.sum() == 0:
        px0, py0, px1, py1 = 0, 0, w, h
    else:
        px0, py0, px1, py1 = mask_to_box(pred_mask)

    iou_val = box_iou((px0, py0, px1, py1), gt_box)

    # Aplicar soft-padding del 15%
    bw, bh = px1 - px0, py1 - py0
    pad_x, pad_y = int(bw * PAD_RATIO), int(bh * PAD_RATIO)
    cx0 = max(0, px0 - pad_x)
    cy0 = max(0, py0 - pad_y)
    cx1 = min(w, px1 + pad_x)
    cy1 = min(h, py1 + pad_y)

    # Recortar y guardar
    crop = img[cy0:cy1, cx0:cx1]
    cv2.imwrite(str(crops_dir / f"{iid}.jpg"), crop)

    stats.append({
        "image_id": iid,
        "filename": row.filename,
        "fold": row.fold,
        "group_id": row.group_id,
        "diagnosis": row.diagnosis,
        "pred_x0": px0, "pred_y0": py0, "pred_x1": px1, "pred_y1": py1,
        "gt_x0": gt_box[0], "gt_y0": gt_box[1], "gt_x1": gt_box[2], "gt_y1": gt_box[3],
        "crop_x0": cx0, "crop_y0": cy0, "crop_x1": cx1, "crop_y1": cy1,
        "box_iou": round(iou_val, 4)
    })

df_stats = pd.DataFrame(stats)
df_stats.to_csv("results/eda/oof_crop_iou_stats.csv", index=False)

print(f" 1,985 OOF crops guardados en {crops_dir}/")
print(f" Resumen Box IoU (Predicho vs Ground-Truth):")
print(f"   - Media IoU:   {df_stats['box_iou'].mean():.4f}")
print(f"   - Mediana IoU: {df_stats['box_iou'].median():.4f}")
print(f"   - % con IoU >= 0.80: {(df_stats['box_iou'] >= 0.80).mean()*100:.1f}%")
print(f"   - % con IoU >= 0.90: {(df_stats['box_iou'] >= 0.90).mean()*100:.1f}%")

 1,985 OOF crops guardados en results/oof_crops/
 Resumen Box IoU (Predicho vs Ground-Truth):
   - Media IoU:   0.8642
   - Mediana IoU: 0.8959
   - % con IoU >= 0.80: 81.9%
   - % con IoU >= 0.90: 48.0%


getting everything in a zip(to not lose if session ends)

In [7]:
import shutil
from pathlib import Path
from google.colab import files

# 1. Verificar que los 3 entregables existen y tienen los 1,985 archivos
n_masks = len(list(Path("results/pred_masks/deeplabv3p_r50").glob("*.png")))
n_crops = len(list(Path("results/oof_crops").glob("*.jpg")))
assert n_masks == 1985, f"Faltan máscaras: {n_masks}"
assert n_crops == 1985, f"Faltan crops: {n_crops}"

# 2. Crear una carpeta limpia con el paquete de entrega de Phase 1 (Member 1)
handoff = Path("phase1_member1_handoff")
handoff.mkdir(exist_ok=True)

shutil.copy("results/eda/folds_master.csv", handoff / "folds_master.csv")
shutil.copy("results/eda/oof_crop_iou_stats.csv", handoff / "oof_crop_iou_stats.csv")
shutil.copy("results/segmentation/deeplabv3p_r50/per_image.csv", handoff / "deeplabv3p_r50_per_image_metrics.csv")
shutil.copytree("results/pred_masks/deeplabv3p_r50", handoff / "oof_binary_masks", dirs_exist_ok=True)
shutil.copytree("results/oof_crops", handoff / "oof_kidney_crops", dirs_exist_ok=True)

# 3. Comprimir en ZIP y descargar a tu ordenador
zip_path = shutil.make_archive("Phase1_Member1_Deliverables", "zip", handoff)
print(f"Paquete listo ({n_masks} máscaras + {n_crops} crops + CSVs): {zip_path}")
files.download(zip_path)

Paquete listo (1985 máscaras + 1985 crops + CSVs): /content/Project-data-science/Phase1_Member1_Deliverables.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

or in drive

In [6]:
from google.colab import drive
import shutil
from pathlib import Path

# 1. Conectar tu Google Drive (te pedirá permiso en una ventana emergente)
drive.mount('/content/drive')

# 2. Copiar el ZIP entero a la carpeta principal de tu Google Drive
dest_dir = Path('/content/drive/MyDrive/Kidney_Phase1_Member1')
dest_dir.mkdir(parents=True, exist_ok=True)

shutil.copy('Phase1_Member1_Deliverables.zip', dest_dir / 'Phase1_Member1_Deliverables.zip')
shutil.copy('results/eda/folds_master.csv', dest_dir / 'folds_master.csv')
shutil.copy('results/eda/oof_crop_iou_stats.csv', dest_dir / 'oof_crop_iou_stats.csv')
shutil.copy('results/segmentation/deeplabv3p_r50/per_image.csv', dest_dir / 'deeplabv3p_r50_per_image_metrics.csv')

print("Todo guardado a salvo en tu Google Drive en: MyDrive/Kidney_AI_Phase1_Member1/")

Mounted at /content/drive
Todo guardado a salvo en tu Google Drive en: MyDrive/Kidney_AI_Phase1_Member1/
